In [7]:
import copy

# Imports
import matplotlib.pyplot as plt
import numpy as np
import glob
import pandas as pd
import os

import ray
from ray.rllib.algorithms.ppo import PPOConfig
from ray.rllib import Policy

import IPython.core.display_functions

from src.parsers import HMParser, CotevParser
from src.resources import Aggregator, Generator, Load, Storage
from src.algorithms.rl import EnergyCommunityContributionPriorityV0

import warnings
warnings.filterwarnings('ignore')

In [2]:
# Data parsing

# EC data for non-renewable generators and batteries
data_ec = HMParser(file_path='/Users/ecgomes/DataspellProjects/pyecom/data/EC_V4.xlsx', ec_id=1)
data_ec.parse()

# EV data from the EV4EU simulator
data_ev = CotevParser(population_path=
                      '/Users/ecgomes/DataspellProjects/pyecom/data/simulation_dataframes_2years/population_731.csv',
                      driving_history_path=
                      '/Users/ecgomes/DataspellProjects/pyecom/data/simulation_dataframes_2years/ev_driving_history_731.csv',
                      assigned_segments_path='/Users/ecgomes/DataspellProjects/pyecom/data/simulation_dataframes_2years/assigned_segments_731.csv',
                      parse_date_start='2019',
                      parse_date_end='2020')
data_ev.parse()

In [3]:
# UPAC Data load

data_upacs = {}
for i in glob.glob('/Users/ecgomes/Documents/PhD/UPAC data/upac*_pv.csv'):
    temp = pd.read_csv(i, index_col=0, parse_dates=True)
    temp = temp.resample('H').mean()

    # Need to divide by 1000 to convert from W to kW
    temp['pv'] = temp['pv'] / 1000
    temp['load'] = temp['load'] / 1000

    # Set any negative values to zero
    temp.loc[temp['pv'] < 0, 'pv'] = 0
    temp.loc[temp['load'] < 0, 'load'] = 0

    # We only want 2019 and 2020 data
    temp = temp.loc['2019':'2020']

    # Fill NaN values with zeros
    temp = temp.fillna(0)

    name = i.split('/')[-1].split('_')[0].split('upac')[1]

    data_upacs[name] = temp

In [4]:
# Train resource creation

def create_resources(upacs, ec, ev):
    """
    Create the resources for the training environment.
    return a list of resources.
    :param upacs: dict with the UPAC data
    :param ec: dict with the EC data
    :param ev: dict with the EV data
    """

    resources = []
    # Add generators (from pv column from the UPAC data)
    for i in range(len(upacs)):
        current_name = list(upacs.keys())[i]
        resources.append(Generator(
            name='ren_generator_' + current_name,
            value=np.zeros(upacs[current_name]['pv'].shape),
            lower_bound=np.zeros(upacs[current_name]['pv'].shape),
            upper_bound=upacs[current_name]['pv'].values,
            cost=ec.generator['cost_parameter_b'][0, 0] * np.ones(upacs[current_name].shape[0]),
            cost_nde=np.tile(ec.generator['cost_nde'][0], (int(upacs[current_name].shape[0] / 24))),
            is_renewable=True))

    # Add loads (from load column from the UPAC data)
    for i in range(len(upacs)):
        current_name = list(upacs.keys())[i]
        resources.append(Load(
            name='load_' + current_name,
            value=upacs[current_name]['load'],
            lower_bound=np.zeros(upacs[current_name].shape),
            upper_bound=upacs[current_name]['load'].values,
            cost=np.ones(upacs[current_name].shape[0]),
            cost_cut=np.tile(ec.load['cost_cut'][0], (int(upacs[current_name].shape[0] / 24))),
            cost_reduce=np.tile(ec.load['cost_reduce'][0], (int(upacs[current_name].shape[0] / 24))),
            cost_ens=np.tile(ec.load['cost_ens'][0], (int(upacs[current_name].shape[0] / 24)))))

    # Add storage (from the EC data)
    for i in range(ec.storage['p_charge_limit'].shape[0]):
        resources.append(Storage(
            name='storage_{:02d}'.format(i+1),
            value=ec.storage['initial_state'][i] * np.ones(upacs['02'].shape[0]),
            lower_bound=np.ones(upacs['02'].shape[0]) * ec.storage['energy_min_percentage'][i],
            upper_bound=(ec.storage['energy_capacity'][i] * np.ones(upacs['02'].shape[0])),
            cost=np.ones(upacs['02'].shape[0]) * 0,
            cost_discharge=np.tile(ec.storage['discharge_price'][i], (int(upacs['02'].shape[0] / 24))),
            cost_charge=np.tile(ec.storage['charge_price'][i], (int(upacs['02'].shape[0] / 24))),
            capacity_max=ec.storage['energy_capacity'][i],
            capacity_min=ec.storage['energy_min_percentage'][i],
            initial_charge=ec.storage['initial_state'][i],
            discharge_efficiency=ec.storage['discharge_efficiency'][i],
            discharge_max=np.tile(ec.storage['p_discharge_limit'][i], (int(upacs['02'].shape[0] / 24))),
            charge_efficiency=ec.storage['charge_efficiency'][i],
            charge_max=np.tile(ec.storage['p_charge_limit'][i], (int(upacs['02'].shape[0] / 24))),
            capital_cost=np.array([0.05250, 0.10500, 0.01575])))

    # Add vehicles (from the EV data)
    for i in np.arange(len(ev)):
        resources.append(ev[i])

    # Append Aggregator
    resources.append(Aggregator(
        name='aggregator',
        value=np.zeros(upacs['02'].shape[0]),
        lower_bound=np.zeros(upacs['02'].shape[0]),
        upper_bound=np.tile(ec.peers['import_contracted_p_max'][0, 0], (upacs['02'].shape[0])),
        cost=np.tile(ec.peers['buy_price'][0, 0], (upacs['02'].shape[0])),
        imports=np.zeros(upacs['02'].shape[0]),
        exports=np.zeros(upacs['02'].shape[0]),
        import_cost=np.tile(ec.peers['buy_price'][0], (int(upacs['02'].shape[0] / 24))), # * 100,
        export_cost=np.tile(ec.peers['sell_price'][0], (int(upacs['02'].shape[0] / 24))),
        import_max=np.tile(ec.peers['import_contracted_p_max'][0, 0], (int(upacs['02'].shape[0]))),
        export_max=np.tile(ec.peers['export_contracted_p_max'][0, 0], (int(upacs['02'].shape[0])))))

    return resources

In [5]:
# Create resources for the training environment

def iterate_resources(u, c, e, mode='daily'):

    temp = {}

    # Save first key of upac data
    first_key = list(u.keys())[0]

    if mode == 'daily':

        # Loop to iterate over days in the datasets
        for i in np.unique(u[first_key].index.date):
            # Create the resources for the training environment

            date = i.strftime('%Y-%m-%d')

            temp_u = {k: v.loc[date] for k, v in u.items()}
            temp_e = e.create_resources(e.population, e.trips_grid, e.assigned_segments, date)

            temp[date] = create_resources(upacs=temp_u,
                                          ec=c,
                                          ev=temp_e)

    elif mode == 'monthly':

        # Loop to iterate over months in the datasets
        # Need to be careful with different years
        unique_months = np.unique(data_upacs['02'].index.strftime('%Y-%m'))

        for i in unique_months:
            # Create the resources for the training environment
            date = i

            temp_u = {k: v.loc[date] for k, v in u.items()}
            temp_e = e.create_resources(e.population, e.trips_grid, e.assigned_segments, date)

            temp[date] = create_resources(upacs=temp_u,
                                          ec=c,
                                          ev=temp_e)

    elif mode == 'yearly':

        # Loop to iterate over years in the datasets
        unique_years = np.unique(data_upacs['02'].index.strftime('%Y'))

        for i in unique_years:
            # Create the resources for the training environment
            date = i

            temp_u = {k: v.loc[date] for k, v in u.items()}
            temp_e = e.create_resources(e.population, e.trips_grid, e.assigned_segments, date)

            temp[date] = create_resources(upacs=temp_u,
                                          ec=c,
                                          ev=temp_e)

    return temp

dataset_resources = iterate_resources(u=data_upacs, c=data_ec, e=data_ev, mode='monthly')

In [8]:
# Create the environment and check if everything is ok

temp_env = EnergyCommunityContributionPriorityV0(ren_generators=dataset_resources[list(dataset_resources.keys())[0]][:5],
                                                 generators=[],#dataset_resources[list(dataset_resources.keys())[0]][5:7],
                                                 loads=dataset_resources[list(dataset_resources.keys())[0]][5:10],
                                                 storages=dataset_resources[list(dataset_resources.keys())[0]][10:13],
                                                 evs=dataset_resources[list(dataset_resources.keys())[0]][13:-1],
                                                 aggregator=dataset_resources[list(dataset_resources.keys())[0]][-1],
                                                 storage_penalty=1,
                                                 ev_penalty=1,
                                                 balance_penalty=1)
temp_env.reset()
terminations = truncations = {a: False for a in temp_env.agents}
terminations['__all__'] = False
truncations['__all__'] = False
while not terminations['__all__'] and not truncations['__all__']:

    #print(temp_env._get_observations().keys())
    #print(temp_env.executed_agents)
    #print(temp_env.execution_order)

    actions = temp_env.action_space_sample()
    next_obs, rewards, terminations, truncations, infos = temp_env.step(actions)

print('Terminated: {}'.format(terminations['__all__']))

Terminated: True


In [9]:
# Assign policies

gamma_gen = 0.0
gamma_storage = 0.9
gamma_ev = 0.9

def assign_policies(env):
    
    policies = {'ren_08': (None,
                          env.observation_space['ren_generator_08'],
                          env.action_space['ren_generator_08'],
                          PPOConfig.overrides(gamma=gamma_gen)),
                'ren_09': (None,
                           env.observation_space['ren_generator_09'],
                           env.action_space['ren_generator_09'],
                           PPOConfig.overrides(gamma=gamma_gen)),
                'ren_02': (None,
                           env.observation_space['ren_generator_02'],
                           env.action_space['ren_generator_02'],
                           PPOConfig.overrides(gamma=gamma_gen)),
                'ren_06': (None,
                           env.observation_space['ren_generator_06'],
                           env.action_space['ren_generator_06'],
                           PPOConfig.overrides(gamma=gamma_gen)),
                'ren_13': (None,
                           env.observation_space['ren_generator_13'],
                           env.action_space['ren_generator_13'],
                           PPOConfig.overrides(gamma=gamma_gen)),
                'ev_01': (None,
                          env.observation_space['ev_01'],
                          env.action_space['ev_01'],
                          PPOConfig.overrides(gamma=gamma_ev)),
                'ev_02': (None,
                          env.observation_space['ev_02'],
                          env.action_space['ev_02'],
                          PPOConfig.overrides(gamma=gamma_ev)),
                'ev_03': (None,
                          env.observation_space['ev_03'],
                          env.action_space['ev_03'],
                          PPOConfig.overrides(gamma=gamma_ev)),
                'ev_04': (None,
                          env.observation_space['ev_04'],
                          env.action_space['ev_04'],
                          PPOConfig.overrides(gamma=gamma_ev)),
                'ev_05': (None,
                          env.observation_space['ev_05'],
                          env.action_space['ev_05'],
                          PPOConfig.overrides(gamma=gamma_ev)),
                'storage_01': (None,
                               env.observation_space['storage_01'],
                               env.action_space['storage_01'],
                               PPOConfig.overrides(gamma=gamma_storage)),
                'storage_02': (None,
                               env.observation_space['storage_02'],
                               env.action_space['storage_02'],
                               PPOConfig.overrides(gamma=gamma_storage)),
                'storage_03': (None,
                               env.observation_space['storage_03'],
                               env.action_space['storage_03'],
                               PPOConfig.overrides(gamma=gamma_storage)),
                'aggregator': (None,
                               env.observation_space['aggregator'],
                               env.action_space['aggregator'],
                               PPOConfig.overrides(gamma=0.9))}
    
    return policies

In [10]:
# Define the penalties

IMPORT_PENALTY = 1
EXPORT_PENALTY = 1
STORAGE_ACTION_PENALTY = 1
STORAGE_ACTION_REWARD = 5
EV_ACTION_PENALTY = 1
EV_ACTION_REWARD = 5
EV_REQUIREMENT_PENALTY = 2000
BALANCE_PENALTY = 5000

In [12]:
# Get the policy checkpoint - Sequential

from ray.tune import register_env
from ray.train import Checkpoint
from ray.rllib.algorithms.algorithm import Algorithm

ray.shutdown()
ray.init()

checkpoint_path = '/Users/ecgomes/ray_results/PPO_2025-01-07_19-19-49/PPO_EC_Contrib_V0_5cf14_00000_0_2025-01-07_19-19-49/checkpoint_000000'

# We need to register the environment
temp_resources = dataset_resources['2019-01']
env = EnergyCommunityContributionPriorityV0(ren_generators=temp_resources[:5],
                                            generators=[],
                                            loads=temp_resources[5:10],
                                            storages=temp_resources[10:13],
                                            evs=temp_resources[13:-1],
                                            aggregator=temp_resources[-1],
                                            storage_penalty=STORAGE_ACTION_PENALTY,
                                            ev_penalty=EV_REQUIREMENT_PENALTY,
                                            balance_penalty=BALANCE_PENALTY)
register_env("EC_Contrib_V0", lambda config: env)

algo = Algorithm.from_checkpoint(checkpoint_path)

2025-01-13 12:57:44,892	INFO worker.py:1642 -- Started a local Ray instance.
2025-01-13 12:57:45,467	WARNING algorithm_config.py:2578 -- Setting `exploration_config={}` because you set `_enable_rl_module_api=True`. When RLModule API are enabled, exploration_config can not be set. If you want to implement custom exploration behaviour, please modify the `forward_exploration` method of the RLModule at hand. On configs that have a default exploration config, this must be done with `config.exploration_config={}`.
2025-01-13 12:57:45,512	WARNING algorithm_config.py:2578 -- Setting `exploration_config={}` because you set `_enable_rl_module_api=True`. When RLModule API are enabled, exploration_config can not be set. If you want to implement custom exploration behaviour, please modify the `forward_exploration` method of the RLModule at hand. On configs that have a default exploration config, this must be done with `config.exploration_config={}`.
(pid=11715) DeprecationWarning: `DirectStepOptimi

In [19]:
# Run the learned policies

from copy import deepcopy

PATH = '../priority_paper/'

current_storages = [bess.initial_charge for bess in dataset_resources['2019-01'][10:13]]
current_evs = [ev.initial_charge for ev in dataset_resources['2019-01'][13:-1]]

for i in list(dataset_resources.keys()):

    test_resources = deepcopy(dataset_resources[i])

    print('Day: {}'.format(i))

    for bess in np.arange(len(test_resources[10:13])):
        test_resources[10:13][bess].initial_charge = current_storages[bess]

    for ev in np.arange(len(test_resources[13:-1])):
        test_resources[13:-1][ev].initial_charge = current_evs[ev]


    test_env = EnergyCommunityContributionPriorityV0(ren_generators=test_resources[:5],
                                                     generators=[],
                                                     loads=test_resources[5:10],
                                                     storages=test_resources[10:13],
                                                     evs=test_resources[13:-1],
                                                     aggregator=test_resources[-1],
                                                     storage_penalty=STORAGE_ACTION_PENALTY,
                                                     ev_penalty=EV_REQUIREMENT_PENALTY,
                                                     balance_penalty=BALANCE_PENALTY)

    init_state = state = {a: algo.get_policy(a).get_initial_state() for a in ['ren_08', 'ren_09', 'ren_02', 'ren_06', 'ren_13', 'storage_01', 'storage_02', 'storage_03', 'ev_01', 'ev_02', 'ev_03', 'ev_04', 'ev_05', 'aggregator']}

    obs, info = test_env.reset()

    # Set up the terminations and truncations
    terminations = truncations = {a: False for a in test_env.agents}
    terminations['__all__'] = False
    truncations['__all__'] = False
    
    env_order = test_env.execution_order
    order_history = [env_order]

    while not terminations['__all__'] and not truncations['__all__']:

        current_agent = test_env.execution_order[test_env._current_agent_idx]

        current_policy = 'ren_08' if current_agent.startswith('ren_generator_08') else \
            'ren_09' if current_agent.startswith('ren_generator_09') else \
                'ren_02' if current_agent.startswith('ren_generator_02') else \
                    'ren_06' if current_agent.startswith('ren_generator_06') else \
                        'ren_13' if current_agent.startswith('ren_generator_13') else \
                            'storage_01' if current_agent.startswith('storage_01') else \
                                'storage_02' if current_agent.startswith('storage_02') else \
                                    'storage_03' if current_agent.startswith('storage_03') else \
                                        'ev_01' if current_agent.startswith('ev_01') else \
                                            'ev_02' if current_agent.startswith('ev_02') else \
                                                'ev_03' if current_agent.startswith('ev_03') else \
                                                    'ev_04' if current_agent.startswith('ev_04') else \
                                                        'ev_05' if current_agent.startswith('ev_05') else \
                                                            'aggregator'

        action_dict, new_state, extra = algo.compute_single_action(observation=obs[current_agent],
                                                                   policy_id=current_policy,
                                                                   state=state[current_policy],
                                                                   explore=False)

        state[current_policy] = new_state

        # print(action_dict.keys())

        obs, rewards, terminations, truncations, info = test_env.step({current_agent: action_dict})
        
        if current_agent == 'aggregator' and not terminations['__all__'] and not truncations['__all__']:
            order_history.append(test_env.execution_order)

    current_storages = [bess.value[-1] for bess in test_env.storages]
    current_evs = [ev.value[-1] for ev in test_env.evs]

    pd_results = pd.DataFrame({})
    # Save the renewable generators
    for iter in np.arange(len(test_env.ren_generators)):
        agent_name = test_env.ren_generators[iter].name
        pd_results[agent_name] = test_env.ren_generators[iter].value

    # Save the storages
    for iter in np.arange(len(test_env.storages)):
        agent_name = test_env.storages[iter].name
        pd_results[agent_name] = test_env.storages[iter].value
        pd_results['{}_charge'.format(agent_name)] = test_env.storages[iter].charge
        pd_results['{}_discharge'.format(agent_name)] = test_env.storages[iter].discharge

    # Save the EVs
    for iter in np.arange(len(test_env.evs)):
        agent_name = test_env.evs[iter].name
        pd_results[agent_name] = test_env.evs[iter].value
        pd_results['{}_charge'.format(agent_name)] = test_env.evs[iter].charge
        pd_results['{}_discharge'.format(agent_name)] = test_env.evs[iter].discharge

    # Save the non-renewable generators
    #for iter in np.arange(len(seq_test_env.generators)):
    #    agent_name = seq_test_env.generators[iter].name
    #    pd_results[agent_name] = seq_test_env.generators[iter].value

    pd_results['imports'] = test_env.aggregator.imports
    pd_results['exports'] = test_env.aggregator.exports

    pd_results['energy_history'] = test_env.energy_history
    
    pd_results['order_history'] = order_history

    if not os.path.exists(PATH):
        os.makedirs(PATH)

    pd_results.to_csv('{}{}.csv'.format(PATH, i))

Day: 2019-01
Day: 2019-02
Day: 2019-03
Day: 2019-04
Day: 2019-05
Day: 2019-06
Day: 2019-07
Day: 2019-08
Day: 2019-09
Day: 2019-10
Day: 2019-11
Day: 2019-12
Day: 2020-01
Day: 2020-02
Day: 2020-03
Day: 2020-04
Day: 2020-05
Day: 2020-06
Day: 2020-07
Day: 2020-08
Day: 2020-09
Day: 2020-10
Day: 2020-11
Day: 2020-12
